In [ ]:
import pandas as pd
import numpy as np

### Load the data

In [ ]:
cubo = pd.read_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\cross selling\data\cubo_1.xlsx", 
                    sheet_name = 'SKUs',
                    skiprows=4
                    )

In [ ]:
ventasfarma = pd.read_csv(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\cross selling\data\ventas_01_03-25.csv")

In [ ]:
ventasamkt = pd.read_csv(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\cross selling\data\DATOS DE VENTA DE ENE-MAR_ 2025 amarke.csv", sep=";")

In [ ]:
ventasamkt.shape

Contamos con el registro de 2,214,604 facturas en la UNE FARMACORP

In [ ]:
clusters = pd.read_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\data\clusters_bdgs.xlsx")

In [ ]:
#convertimos fechas a formato datetime
ventasfarma['FECHA_FACT']= pd.to_datetime(ventasfarma['FECHA_FACT'])

In [ ]:
#visualozamos las ventas por dia
ventasxdia = ventasfarma.FECHA_FACT.value_counts().reset_index()

In [ ]:
ventasxdia.columns = ['FECHA_FACT', 'CANT_VENTAS']

In [ ]:
ventasxdia

In [ ]:
#analizamos el dia com mas movimiento (SABADO antes de Carnavales)
ventasxfact = ventasfarma[(ventasfarma['FECHA_FACT'] == '2025-03-01 00:00:00')]

In [ ]:
#ventasxfact = ventasxfact[(ventasxfact['FECHA_FACT'] >= '2025-03-01 00:00:00') & (ventasxfact['FECHA_FACT'] <= '2025-03-05 00:00:00')]

In [ ]:
ventasxfact.shape

### Procesamos la data

In [ ]:
cubo['COD_ARTICULO'] = cubo['COD_ARTICULO'].astype(str)
ventasxfact['COD_ARTICULO'] = ventasxfact['COD_ARTICULO'].astype(str)

In [ ]:
# filtering only SKUs on FARMACORP
ventas_farmacia = pd.merge(cubo,
                            ventasxfact,
                            on='COD_ARTICULO',
                            how='inner'
                            )

In [ ]:
ventas_farmacia['CAT 0'].value_counts()

In [ ]:
# Evaluamos OTC solo
ventas_solofarma = ventas_farmacia[ventas_farmacia['CAT 0'].isin(['ETICOS', 'OTC'])]

In [ ]:
#vamos a filtrar solo BODEGAS FARMACORP para este analisis
clusters_fc = clusters[clusters['UNE']=='FARMACORP']

In [ ]:
#base sobre la cual se va a trabajar
ventas_farmacia = pd.merge(
    ventas_solofarma,
    clusters_fc,
    left_on='COD_BODEGA',
    right_on='BODEGA',
)

In [ ]:
ventas_farmacia.shape

In [ ]:
ventas_group = ventas_farmacia.pivot_table(
    index=['NUMERO_FACTURA'],
    columns=['CAT 4'],
    values='UNIDADES',
    aggfunc='sum'
).reset_index()

In [ ]:
# Convertir a binario (compró o no compró cada categoría)
ventas_binario = (ventas_group.drop('NUMERO_FACTURA', axis=1) > 0).astype(int)

In [ ]:
# Correlación sobre compra sí/no
correlacion_binaria = ventas_binario.corr()

In [ ]:
# Crear máscara para el triángulo superior (evitar duplicados)
mask = np.triu(np.ones(correlacion_binaria.shape), k=1).astype(bool)
correlacion_valores = correlacion_binaria.where(mask)


In [ ]:
# Top 15 pares de categorías más relacionadas
top_pares = correlacion_valores.unstack().sort_values(ascending=False).head(20)
top_pares

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Heatmap de correlación binaria
plt.figure(figsize=(14, 12))
sns.heatmap(correlacion_binaria, 
            annot=True, 
            fmt='.2f',
            cmap='YlOrRd',
            square=True,
            linewidths=0.5,
            cbar_kws={'label': 'Correlación de co-compra'})
plt.title('Relación entre Categorías (Co-compra)', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
import networkx as nx

# Filtrar solo correlaciones fuertes (>0.3)
G = nx.Graph()

for cat1 in correlacion_binaria.columns:
    for cat2 in correlacion_binaria.columns:
        if cat1 < cat2:  # Evitar duplicados
            corr = correlacion_binaria.loc[cat1, cat2]
            if corr > 0.1:  # Umbral de correlación fuerte
                G.add_edge(cat1, cat2, weight=corr)

# Visualizar
plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, k=2, iterations=50)
edges = G.edges()
weights = [G[u][v]['weight'] * 5 for u, v in edges]

nx.draw(G, pos, 
        with_labels=True, 
        node_color='lightblue',
        node_size=3000,
        font_size=10,
        font_weight='bold',
        width=weights,
        edge_color='gray')
plt.title('Red de Categorías Relacionadas (correlación > 0.3)', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
correlacion = ventas_group.drop('NUMERO_FACTURA', axis=1).corr()
print(correlacion)

# O para verlo más ordenado
print(correlacion.round(2))

In [ ]:
# Extraer pares de correlación (excluyendo la diagonal)
import numpy as np

correlacion_valores = correlacion.where(np.triu(np.ones(correlacion.shape), k=1).astype(bool))
correlacion_ordenada = correlacion_valores.unstack().sort_values(ascending=False)

print("Top 10 correlaciones más altas:")
correlacion_ordenada.head(20).round(2)

In [ ]:
ventas_group = ventas_farmacia.groupby(['CAT 0'])['NUMERO_FACTURA'].count().reset_index().sort_values(by='NUMERO_FACTURA', ascending=False)

### Calculamos medidas

In [ ]:
#calculamos el total de facturas emitidas en el rango de fechas de los datos
total_fact = ventas_farmacia['NRO_FACTURAS'].nunique()
total_fact

In [ ]:
#contamos facturas en las que aparece al menos 1 vez cada SKU
ventas_sku = ventas_farmacia.groupby(
    ['COD_ARTICULO', 'ARTICULO']
    )['NRO_FACTURAS'].nunique().reset_index().sort_values(by='NRO_FACTURAS', ascending=False)

In [ ]:
#calculamos el peso de cada SKU sobre el total de las facturas
ventas_sku['weight'] = (ventas_sku['NRO_FACTURAS']/total_fact).round(8)

In [ ]:
#seleccionamos las facturas que solo tienen 1 sku
facturas_unicas = (
    ventas_farmacia
    .groupby(['NRO_FACTURAS'])
    .size()
    .loc[lambda x: x == 1]
    .reset_index()[['NRO_FACTURAS']]
)

#filtramos la data con los sku que aparecen solos en 1 factura
ventas_solo = ventas_farmacia.merge(
    facturas_unicas,
    on=['NRO_FACTURAS'],
    how='inner'
)

In [ ]:
cronicos_solos = ventas_solo[ventas_solo['CAT 1']=='ETICOS CRONICOS']
cronicos_solos.NRO_FACTURAS.nunique()

In [ ]:
skincare_solos = ventas_solo[ventas_solo['CAT 2']=='SKIN CARE']
skincare_solos.NRO_FACTURAS.nunique()

In [ ]:
skincare_solos

In [ ]:
maquillaje.ARTICULO.value_counts().head(30)

In [ ]:
agudos_solos = ventas_solo[ventas_solo['CAT 1']=='ETICOS AGUDOS']
agudos_solos.NRO_FACTURAS.nunique()

In [ ]:
tratamiento_solos = ventas_solo[ventas_solo['CAT 1']=='ETICOS TRATAMIENTO']
tratamiento_solos.NRO_FACTURAS.nunique()

In [ ]:
#contamos facturas con un solo sku
total_fact_solo = ventas_solo['NRO_FACTURAS'].nunique()
total_fact_solo

In [ ]:
#calculamos el % que representa del total de facturas
fact_solo = total_fact_solo/total_fact
fact_solo

In [ ]:
#contamos cuantas veces aparece cada sku solo 
ventas_sku_solo = ventas_solo.groupby(
    ['COD_ARTICULO', 'ARTICULO']
    )['NRO_FACTURAS'].count().reset_index().sort_values(by='NRO_FACTURAS', ascending=False)

In [ ]:
ventas_sku_solo

El SKU que se vende mas solo es Quetorol, seguido de Zopiclona, Migranol y Typirec

In [ ]:
#calculamos el peso de cada sku en las facturas solas
ventas_sku_solo['weight'] = ventas_sku_solo['NRO_FACTURAS']/total_fact_solo

In [ ]:
#unimos todo
skus_analysis = pd.merge(
    ventas_sku,
    ventas_sku_solo,
    on='COD_ARTICULO',
    how='left',
    suffixes=['_gral', '_alone']
)

In [ ]:
#calculamos el % en peso de las veces que cada sku aparece solo vs el total de veces que aparece en una factura
skus_analysis['weight_overeach'] = skus_analysis['NRO_FACTURAS_alone']/skus_analysis['NRO_FACTURAS_gral']

In [ ]:
skus_analysis['weight_overall'] = skus_analysis['NRO_FACTURAS_alone']/total_fact

In [ ]:
skus_analysis.columns

In [ ]:
skus_analysis = skus_analysis[[
    'COD_ARTICULO', 'ARTICULO_gral', 
    'NRO_FACTURAS_gral', 'NRO_FACTURAS_alone', 
    'weight_gral','weight_alone',
    'weight_overeach', 'weight_overall'
]].copy()

NRO_FACTURAS_gral: número total de facturas en las que el SKU aparece al menos una vez, independientemente de si la factura contiene otros SKUs.

NRO_FACTURAS_alone: número de facturas en las que el SKU aparece de forma exclusiva, es decir, facturas que contienen únicamente ese SKU y ningún otro.

weight_gral: proporción de facturas en las que aparece el SKU respecto al total de facturas emitidas. Mide la presencia general del SKU en el conjunto completo de facturación.

weight_alone: proporción de facturas de un solo SKU en las que aparece el SKU, respecto al total de facturas que contienen únicamente un SKU. Refleja la participación del SKU dentro de las facturas unitarias.

weight_overeach: proporción de facturas en las que el SKU aparece solo respecto al total de facturas en las que dicho SKU aparece al menos una vez. Indica qué tan frecuentemente el SKU se vende de manera exclusiva cuando está presente en una factura.

weight_overall: proporción de facturas en las que el SKU aparece solo respecto al total de facturas emitidas. Representa el peso absoluto de las ventas exclusivas del SKU sobre toda la facturación.

In [ ]:
skus_analysis = skus_analysis[skus_analysis['NRO_FACTURAS_alone']>=1000]

In [ ]:
skus_analysis = skus_analysis.sort_values(
    by=['weight_gral'], 
    ascending=[False]).round(5)

In [ ]:
#skus_analysis.to_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\output\analysis_sku.xlsx", index=False)

In [ ]:
ventas_farmacia.columns

### Searching the AyB best pairs for top 500

In [ ]:
# selfmerge for finding pairs of CAT 4 on same invoice
pairs_factura = pd.merge(
    ventas_farmacia,
    ventas_farmacia,
    on=["NRO_FACTURAS"],
    suffixes=("_A", "_B")
)

In [ ]:
pairs_factura.columns

In [ ]:
# filtering combinations to avoid duplicates
pairs_factura = pairs_factura[pairs_factura["ARTICULO_A"] != pairs_factura["ARTICULO_B"]]

# keeping only one order of each pair
pairs_factura = pairs_factura[pairs_factura["ARTICULO_A"] > pairs_factura["ARTICULO_B"]]

# keeping only one order of each pair
pairs_factura = pairs_factura[pairs_factura["CAT 4_A"] != pairs_factura["CAT 4_B"]]


In [ ]:
ventasABxfac = pairs_factura.groupby([
    'CAT 1_A', 'CAT 2_A', 'CAT 3_A', 'CAT 4_A', 'COD_ARTICULO_A', 'ARTICULO_A',
    'CAT 1_B', 'CAT 2_B', 'CAT 3_B', 'CAT 4_B', 'COD_ARTICULO_B', 'ARTICULO_B'
]
    ).agg({'NRO_FACTURAS':'nunique'}).reset_index()

In [ ]:
ventasABxfac = ventasABxfac.rename(columns={'NRO_FACTURAS' : 'NRO_FACTURAS_AB'})

In [ ]:
ventasABxfac.sort_values(by=['NRO_FACTURAS_AB'], ascending=[False])

In [ ]:
ventasABxfac['weight_AB'] = ventasABxfac['NRO_FACTURAS_AB'] / total_fact

In [ ]:
ventasxpairs = pd.merge(
    ventasABxfac,
    ventas_sku,
    left_on='COD_ARTICULO_A',
    right_on='COD_ARTICULO'
)

In [ ]:
ventasxpairs.columns

In [ ]:
ventasxpairs_final = pd.merge(
    ventasxpairs[[
        'CAT 1_A', 'CAT 2_A', 'CAT 3_A', 'CAT 4_A', 'COD_ARTICULO_A', 'ARTICULO_A', 
        'CAT 1_B', 'CAT 2_B', 'CAT 3_B', 'CAT 4_B', 'COD_ARTICULO_B', 'ARTICULO_B', 
        'NRO_FACTURAS_AB', 'weight_AB', 'NRO_FACTURAS', 'weight']],
    ventas_sku,
    left_on='COD_ARTICULO_B',
    right_on='COD_ARTICULO'
)

In [ ]:
ventasxpairs_final = ventasxpairs_final.rename(columns={'weight_x':'weight_A',
                                                        'NRO_FACTURAS_x':'NRO_FACTURAS_A',
                                                        'weight_y':'weight_B',
                                                        'NRO_FACTURAS_y':'NRO_FACTURAS_B'})

In [ ]:
ventasxpairs_final.columns

In [ ]:
ventasxpairs_final = ventasxpairs_final[[
    'CAT 1_A', 'CAT 2_A', 'CAT 3_A', 'CAT 4_A', 'COD_ARTICULO_A', 'ARTICULO_A', 
    'CAT 1_B', 'CAT 2_B', 'CAT 3_B', 'CAT 4_B', 'COD_ARTICULO_B', 'ARTICULO_B', 
    'NRO_FACTURAS_A', 'weight_A', 
    'NRO_FACTURAS_B', 'weight_B',
    'NRO_FACTURAS_AB', 'weight_AB'
]]

In [ ]:
ventasxpairs_final = ventasxpairs_final.sort_values(
    by=['COD_ARTICULO_A',
        'NRO_FACTURAS_AB','weight_AB'], 
        ascending=[False, False, False])

In [ ]:
ventasxpairs_final

In [ ]:
top_sku_pairs = ventasxpairs_final.groupby('COD_ARTICULO_A').head(10).reset_index(drop=True)

In [ ]:
top_sku_pairs

### Getting the final base for analysis

In [ ]:
base_analysis = pd.merge(
    top_sku_pairs,
    skus_analysis,
    left_on='COD_ARTICULO_A',
    right_on='COD_ARTICULO',
    how='right')

In [ ]:
cubo.columns

In [ ]:
base = pd.merge(
    base_analysis,
    cubo[['COD_ARTICULO', 'Precio Unitario FA', 'CR Unitario']],
    left_on='COD_ARTICULO_B',
    right_on='COD_ARTICULO',
    how='left'
)

In [ ]:
base_cat1 = pd.merge(
    base,
    cubo[['COD_ARTICULO', 'CAT 1']],
    left_on='COD_ARTICULO_A',
    right_on='COD_ARTICULO',
    how='left'
)

In [ ]:
base_cat1

In [ ]:
base_cat1[['CAT 4_A', 'CAT 4_B']].value_counts()

In [ ]:
base_final = base_cat1[[
    'CAT 1',
    'COD_ARTICULO_A', 'ARTICULO_A', 
    'COD_ARTICULO_B', 'ARTICULO_B',
    
    'NRO_FACTURAS_A', 'weight_A',
    'NRO_FACTURAS_B', 'weight_B',
    'NRO_FACTURAS_AB', 'weight_AB',
    
    'NRO_FACTURAS_alone', 'weight_alone', 
    'weight_overeach', 'weight_overall',
    
    'Precio Unitario FA',
    'CR Unitario'
]]

In [ ]:
base_final = base_final.rename(columns={
    'Precio Unitario FA': 'PVU_B',
    'CR Unitario' : 'CRU_B'
    })

In [ ]:
base_final['Profit_B'] = base_final['PVU_B'] - base_final['CRU_B']

In [ ]:
base_final.head(10)

In [ ]:
profit_top_sku = (
    base_final
    .groupby('COD_ARTICULO_A', as_index=False)['Profit_B']
    .mean()
    .rename(columns={'Profit_B': 'Profit_B_avg'})
)

In [ ]:
base_final = pd.merge(
    base_final,
    profit_top_sku,
    on='COD_ARTICULO_A'
)

In [ ]:
base_final['ambition'] = 0.05

In [ ]:
base_final['opportunity'] = base_final['ambition'] * base_final['NRO_FACTURAS_alone'] * base_final['Profit_B']

In [ ]:
base_final['opportunity_avg'] = base_final['ambition'] * base_final['NRO_FACTURAS_alone'] * base_final['Profit_B_avg']

In [ ]:
base_final

In [ ]:
base_final.to_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\output\base_analysis_sku.xlsx",
                     index=False)

In [ ]:
# Top 50 ARTICULO_A by NRO_FACTURAS_A
if 'NRO_FACTURAS_A' not in base.columns:
    raise KeyError("'NRO_FACTURAS_A' not found in `base`. Check previous merges.")

group_cols = ['COD_ARTICULO_A', 'ARTICULO_A']

top50_articulo_A = (
    base
    .groupby(group_cols, as_index=False)['NRO_FACTURAS_A']
    .max()
    .sort_values(by='NRO_FACTURAS_A', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

# attach weight_A if available
if 'weight_A' in base.columns:
    top50_articulo_A = top50_articulo_A.merge(
        base.groupby(group_cols, as_index=False)['weight_A'].max(),
        on=group_cols,
        how='left'
    )


In [ ]:

# display
top50_articulo_A

In [ ]:
# Top 50 ARTICULO_B by NRO_FACTURAS_B (including partners count)
if 'NRO_FACTURAS_B' not in base.columns:
    raise KeyError("'NRO_FACTURAS_B' not found in `base`. Check previous merges.")

group_cols_b = ['COD_ARTICULO_B', 'ARTICULO_B']

# aggregate invoices per ARTICULO_B
agg_b = (
    base
    .groupby(group_cols_b, as_index=False)
    .agg(NRO_FACTURAS_B=('NRO_FACTURAS_B', 'max'))
)

# count distinct partners (COD_ARTICULO_A) per ARTICULO_B
partners = (
    base
    .groupby(group_cols_b, as_index=False)
    .agg(partners_count=('COD_ARTICULO_A', 'nunique'))
)

# merge results
top50_articulo_B = agg_b.merge(partners, on=group_cols_b, how='left')

# attach weight_B if available
if 'weight_B' in base.columns:
    w = base.groupby(group_cols_b, as_index=False)['weight_B'].max()
    top50_articulo_B = top50_articulo_B.merge(w, on=group_cols_b, how='left')

# sort and take top 50
top50_articulo_B = (
    top50_articulo_B
    .sort_values(by='NRO_FACTURAS_B', ascending=False)
    .head(100)
    .reset_index(drop=True)
)


In [ ]:
# display
top50_articulo_B